# 03 — Lexicon Tier

Builds the project's **multi-dimensional desire signature** from non-trainable lexicons:

- VADER compound + (pos / neg / neu) — polarity axis
- AFINN — independent polarity baseline
- (Optional) NRC EmoLex — 8 Plutchik emotions
- (Optional) NRC VAD — continuous valence / arousal / dominance

NRC lexicons require a separate download (free for research) from <https://saifmohammad.com/WebPages/NRC-Emotion-Lexicon.htm>. Drop the txt files into `data/raw/nrc/` and the cells below will pick them up. Without them, the pipeline still works — just a smaller signature.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from utils import ARTIFACTS_DIR, RAW_DIR, FIGURES_DIR, save_metrics, set_seed
set_seed()
sns.set_theme(style="whitegrid")

In [ ]:
# Use the test split for honest reporting
df = pl.read_parquet(ARTIFACTS_DIR / "split_test.parquet").to_pandas()
print(df.shape)
df.head(2)

In [ ]:
import nltk
try:
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
    sid = SentimentIntensityAnalyzer()
except LookupError:
    nltk.download("vader_lexicon")
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
    sid = SentimentIntensityAnalyzer()

vader = pd.DataFrame([sid.polarity_scores(t) for t in df["review"]])
vader.columns = [f"vader_{c}" for c in vader.columns]
print(vader.head(2))

In [ ]:
from afinn import Afinn
afinn = Afinn()
df["afinn"] = [afinn.score(t) for t in df["review"]]
df["afinn_norm"] = df["afinn"] / (df["review"].str.split().str.len().clip(lower=1))
print(df[["afinn", "afinn_norm"]].describe())

In [ ]:
# Optional NRC EmoLex
NRC_EMO = RAW_DIR / "nrc" / "NRC-Emotion-Lexicon-Wordlevel-v0.92.txt"
emo_cols = []
if NRC_EMO.exists():
    nrc = pd.read_csv(NRC_EMO, sep="\t", header=None, names=["word", "emotion", "value"])
    nrc = nrc[nrc["value"] == 1].pivot_table(index="word", columns="emotion", values="value", fill_value=0)
    emo_cols = list(nrc.columns)
    print("Loaded NRC EmoLex:", nrc.shape, "columns:", emo_cols)

    def emo_vec(text):
        toks = text.split()
        if not toks:
            return np.zeros(len(emo_cols))
        sub = nrc.reindex(toks).fillna(0)
        return sub.mean(axis=0).values

    emo_mat = np.vstack([emo_vec(t) for t in df["review"]])
    emo_df = pd.DataFrame(emo_mat, columns=[f"emo_{c}" for c in emo_cols], index=df.index)
else:
    print(f"NRC EmoLex not found at {NRC_EMO} — skipping (still fine).")
    emo_df = pd.DataFrame(index=df.index)

In [ ]:
# Combine into the per-review desire signature
sig = pd.concat(
    [df[["label", "review"]].reset_index(drop=True),
     vader.reset_index(drop=True),
     df[["afinn", "afinn_norm"]].reset_index(drop=True),
     emo_df.reset_index(drop=True)],
    axis=1,
)
sig.head(3)

In [ ]:
# Class-conditional means heatmap: shows which signature dims separate pos/neg
numeric_cols = [c for c in sig.columns if c not in ("label", "review")]
class_means = sig.groupby("label")[numeric_cols].mean().T
class_means.columns = ["negative (0)", "positive (1)"]

fig, ax = plt.subplots(figsize=(8, max(4, len(numeric_cols) * 0.3)))
sns.heatmap(class_means, annot=True, fmt=".3f", cmap="vlag", center=0, ax=ax)
ax.set_title("Mean signature value by polarity class")
plt.tight_layout()
fig_path = FIGURES_DIR / "03_lexicon_signature_means.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved {fig_path}")

In [ ]:
# Quick lexicon baseline accuracy: predict positive iff vader_compound > 0
preds = (sig["vader_compound"] > 0).astype(int).values
acc = (preds == sig["label"].values).mean()
print(f"VADER threshold @ 0: accuracy = {acc:.4f}")

save_metrics("lexicon_vader", {"accuracy": float(acc), "threshold": 0.0})
sig.to_parquet(ARTIFACTS_DIR / "lexicon_signature_test.parquet")
print("Saved lexicon_signature_test.parquet")

## Findings to record in Methodology / Results

- VADER-only accuracy on test split (above) is the **lexicon-tier baseline**. The classical-ML and transformer tiers should beat it.
- The class-mean heatmap shows which lexicon dimensions are most discriminative — keep it for the LaTeX Methodology figure.
- Without NRC, the signature is 6-dim; with NRC EmoLex, 14-dim; add NRC VAD for 17-dim.